# Multilayer Perceptrons
You should build an end-to-end machine learning pipeline using a multilayer perceptron model. In particular, you should do the following:
- Load the `mnist` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Split the dataset into training and test sets using [Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
- Build an end-to-end machine learning pipeline, including a [multilayer perceptron](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) model.
- Optimize your pipeline by validating your design decisions.
- Test the best pipeline on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/m-mahdavi/teaching/refs/heads/main/datasets/mnist.csv')
df.head()

,id,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,31953,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,34452,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,60897,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,36953,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1981,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Shape of original data:- {df.shape}")
print(f"Shape of train data:- {df_train.shape}")
print(f"Shape of test data:- {df_test.shape}")

Shape of original data:- (4000, 786)
Shape of train data:- (3200, 786)
Shape of test data:- (800, 786)


In [4]:
x_train = df_train.drop(["id", "class"], axis=1,)
y_train = df_train["class"]

x_test = df_test.drop(["id", "class"], axis=1)
y_test = df_test["class"]

print(f"Shape of x_train:- {x_train.shape}")
print(f"Shape of x_test:- {x_test.shape}")
print(f"Shape of y_train:- {y_train.shape}")
print(f"Shape of y_test:- {y_test.shape}")

Shape of x_train:- (3200, 784)
Shape of x_test:- (800, 784)
Shape of y_train:- (3200,)
Shape of y_test:- (800,)


In [5]:
numerical_attributes = x_train.select_dtypes(include=['int64']).columns
pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([("scaling", StandardScaler(), numerical_attributes)])),
    ('classifier', MLPClassifier())
])

In [6]:
mlp_param = {
    'classifier__hidden_layer_sizes': [(80,), (100,), (120,), (100,100), (100,50)],
    'classifier__activation': ['relu', 'logistic'],
    'classifier__solver': ['adam', 'sgd'],
    'classifier__alpha': [0.0001, 0.001, 0.01],
    'classifier__learning_rate': ['constant', 'adaptive'],
    'classifier__learning_rate_init': [0.001, 0.01, 0.1],
    'classifier__max_iter': [50, 100, 200, 300],
    'classifier__batch_size': ['auto']
}

In [8]:
mlp_random = RandomizedSearchCV(estimator = pipeline, param_distributions = mlp_param, n_iter = 5, cv = 5, verbose=2, random_state=42, n_jobs = -1)
mlp_random.fit(x_train, y_train)

print("Best parameters:", mlp_random.best_params_)
print("Best score:", mlp_random.best_score_)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best parameters: {'classifier__solver': 'sgd', 'classifier__max_iter': 300, 'classifier__learning_rate_init': 0.01, 'classifier__learning_rate': 'adaptive', 'classifier__hidden_layer_sizes': (120,), 'classifier__batch_size': 'auto', 'classifier__alpha': 0.001, 'classifier__activation': 'logistic'}
Best score: 0.9103125000000001


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
